In [ ]:
from matplotlib.gridspec import GridSpec
import networkx as nx
from itertools import product
from collections import defaultdict
import scipy as sp
import matplotlib as mpl
from scipy.stats import circmean
from scipy.stats import circstd
from pathlib import Path
import sys
import numpy as np
import glob
import os
from scipy.stats import iqr
from scipy.ndimage import gaussian_filter


%load_ext autoreload
%autoreload

def get_parent_dir():
    try:
        return Path(__file__).resolve().parent.parent
    except NameError:
        return Path.cwd().parent

parent_dir = str(get_parent_dir())
print("Parent directory:", parent_dir)

sys.path.append(parent_dir)
from io_experiments.inhibition_experiments import *

In [ ]:
run_results_subdir = single_DCN (50, duration = 4_000.0, spike_start=2500.0)
filepath_spiking_STO = combine_data(run_results_subdir)
print(f'filepath_spiking_STO = {filepath_spiking_STO!r}')


In [ ]:
run_results_subdir = single_DCN (50, duration = 4_000.0, spike_start = None)
filepath_baseline_STO = combine_data(run_results_subdir)
print(f'filepath_baseline_STO = {filepath_baseline_STO!r}')

In [ ]:
run_results_subdir = single_DCN (50, duration = 4_000.0, spike_start=2500.0, gamma_CN_IO = 0.00)
filepath_spiking_STO_control = combine_data(run_results_subdir)
print(f'filepath_spiking_STO_control = {filepath_spiking_STO_control!r}')

In [ ]:
run_results_subdir = single_DCN (50, duration = 4_000.0, spike_start = None, gamma_CN_IO= 0.00)
filepath_baseline_STO_control = combine_data(run_results_subdir)
print(f'filepath_baseline_STO_control = {filepath_baseline_STO_control!r}')

In [ ]:
# Spiking dataset
filepath_spiking_STO = 'C:\\Users\\HP\\PycharmProjects\\Internproject 2025\\cerebellum-jax-main\\data\\datasets\\single_DCN_inhibition_gamma_CN=-0.02_DCN_spikes=True_n_runs_100_seeds_88-137_combined.npz'

# Baseline dataset
filepath_baseline_STO = 'C:\\Users\\HP\\PycharmProjects\\Internproject 2025\\cerebellum-jax-main\\data\\datasets\\single_DCN_inhibition_gamma_CN=-0.02_DCN_spikes=False_n_runs_100_seeds_88-137_combined.npz'

filepath_spiking_STO_control = 'C:\\Users\\HP\\PycharmProjects\\Internproject 2025\\cerebellum-jax-main\\data\\datasets\\single_DCN_inhibition_gamma_CN=0.0_DCN_spikes=True_n_runs_100_seeds_88-137_combined.npz'

filepath_baseline_STO_control = 'C:\\Users\\HP\\PycharmProjects\\Internproject 2025\\cerebellum-jax-main\\data\\datasets\\single_DCN_inhibition_gamma_CN=0.0_DCN_spikes=False_n_runs_100_seeds_88-137_combined.npz'


filepaths = {
    "default": {
        "spiking": filepath_spiking_STO,
        "baseline": filepath_baseline_STO,
    },
    "control": {
        "spiking": filepath_spiking_STO_control,
        "baseline": filepath_baseline_STO_control,
    },
}

This section:

o For plotting single traces:
- Load spiking and baseline dataset
- Split datasets into connected and unconnected IO network conditions
- Get IO soma traces in window of interest

o For Phase Response Curves:
- Loop through different frequency ranges
- Get phases of IO traces (wrapped + unwrapped)
- In region of interest, filter out non-periodic IO traces
- Get baseline phase at time of CN spike (x-axis)
- For each spike trial, find time after which 1 baseline cycle has passed
- Calculate the difference in phase between baseline and spiking unwrapped phase traces (y-axis)
    (spiking condition - baseline)

- Sort IO spiking data in the same way as soma potential phases
- Find time of first IO spike after CN spike
- Filter out trials without IO spiking after CN spike
- Calculate difference in timing of the first spike (spiking condition - baseline)


o To do: Find origin of pi offsets in delta phase; Does this accurately reflect my data or is it something to remove via modulus 2 pi
- NExt peak measure
- Baseline / before CN spike delta phase
- Different Hilbert


In [ ]:
condition = 'default'

r_spike_dat = np.load(filepaths[condition]['spiking'], allow_pickle=True) # Load spiking data
spike_dat = {item: r_spike_dat[item] for item in r_spike_dat.files}

r_dat = np.load(filepaths[condition]['baseline'], allow_pickle=True) # Load baseline data
dat = {item: r_dat[item] for item in r_dat.files}

# Split into connected and unconnected conditions
spk_connected_run_idx= np.where(spike_dat["IO_n_projections"] == 4)[0]
spk_unconnected_run_idx=  np.where(spike_dat["IO_n_projections"] == 0)[0]
connected_run_idx= np.where(dat["IO_n_projections"] == 4)[0]
unconnected_run_idx=  np.where(dat["IO_n_projections"] == 0)[0]

# Determine window of interest
n_runs = np.int64(len(spike_dat['dt'])/2)
DCN_spike_list = np.where(spike_dat['cn.spike'][0,:,0])[0]

dt = spike_dat['dt'][0]
window_length = 700 #ms
center_CN_spike = DCN_spike_list[0]
half_window_idx = np.int64(np.round(window_length/dt))
window = ( np.int64(center_CN_spike-half_window_idx) , np.int64(center_CN_spike+half_window_idx) )

time =  spike_dat['ts'][0,:]
windowed_time = time[window[0]:window[1]]
DCN_spike_time = spike_dat['ts'][0,DCN_spike_list[0]: DCN_spike_list[-1]]

############# ---------- Single traces ----------------- #################

# CN spiking in connected IO network
spk_single_IO_traces_connected = spike_dat['io.V_soma'][spk_connected_run_idx,:,: ]# shape (nruns, window, n_ios)
spk_con_IO_traces_stacked = np.hstack([ spk_single_IO_traces_connected[run, :, :] for run in np.arange(n_runs)]) # shape (window, nruns * n_ios)

# CN spiking in unconnected IO network
spk_single_IO_traces_unconnected = spike_dat['io.V_soma'][spk_unconnected_run_idx,:,: ]# shape (nruns, window, n_ios)
spk_uncon_IO_traces_stacked = np.hstack([ spk_single_IO_traces_unconnected[run, :, :] for run in np.arange(n_runs)]) # shape (window, nruns * n_ios)

# Connected IO network
single_IO_traces_connected = dat['io.V_soma'][connected_run_idx,:,: ]# shape (nruns, window, n_ios)
con_IO_traces_stacked = np.hstack([single_IO_traces_connected[run, :, :] for run in np.arange(n_runs)]) # shape (window, nruns * n_ios)

# Unconnected IO network
single_IO_traces_unconnected = dat['io.V_soma'][unconnected_run_idx,:,: ]# shape (nruns, window, n_ios)
uncon_IO_traces_stacked = np.hstack([ single_IO_traces_unconnected[run, :, :] for run in np.arange(n_runs)]) # shape (window, nruns * n_ios)

######## ----------- Power spectra --------------- ########

Fs = 1 / (dt / 1000)
data_detrend = np.stack([
    detrend(spike_dat['io.V_soma'], spk_unconnected_run_idx),
    detrend(spike_dat['io.V_soma'], spk_connected_run_idx),
    detrend(dat[ 'io.V_soma'], unconnected_run_idx),
    detrend(dat[ 'io.V_soma'],  connected_run_idx)
    ], axis=0)


freqs, PSD_spk_uncon, kmax= multitaper_psd(data_detrend[0], fs=Fs)
freqs, PSD_spk_con, kmax = multitaper_psd(data_detrend[1], fs=Fs)
freqs, PSD_uncon, kmax =multitaper_psd(data_detrend[2], fs=Fs)
freqs, PSD_con, kmax = multitaper_psd(data_detrend[3], fs=Fs)

######## ----------- Phase Responce Curves --------------- ########

# Hilbert transform for phases and amplitudes
uncon_freqrange = [2, 12]
con_freqrange = [5, 10]

spk_uncon_phi, spk_uncon_amplitudes = filter_and_Hilbert(data_detrend[0] , Fs=Fs, a = uncon_freqrange[0], b = uncon_freqrange[1])
uncon_phi, uncon_amplitudes = filter_and_Hilbert(data_detrend[2], Fs=Fs, a = uncon_freqrange[0], b = uncon_freqrange[1])

spk_con_phi, spk_con_amplitudes = filter_and_Hilbert(data_detrend[1], Fs=Fs , a = con_freqrange[0], b = con_freqrange[1])
con_phi, con_amplitudes = filter_and_Hilbert(data_detrend[3], Fs=Fs, a = con_freqrange[0], b = con_freqrange[1])


# Get unwrapped phases:
spk_uncon_phi_unwrap = np.unwrap(spk_uncon_phi, axis=0) 
uncon_phi_unwrap = np.unwrap(uncon_phi, axis=0)

spk_con_phi_unwrap = np.unwrap(spk_con_phi, axis=0)
con_phi_unwrap = np.unwrap(con_phi, axis=0)


# Exclude IO cells that are very weak oscillators, or have small amplitudes
r_sqr_thresh = 0.99
uncon_include_idx = np.unique(np.concatenate([
                     assess_phase_linearity(spk_uncon_phi_unwrap, time_window = (window[0], center_CN_spike), time=time, r_sqr_thresh = r_sqr_thresh ),
                     assess_phase_linearity(uncon_phi_unwrap, time_window = (window[0], center_CN_spike), time=time, r_sqr_thresh = r_sqr_thresh ),
                    np.where(np.mean(spk_uncon_amplitudes[window[0]:center_CN_spike,:], axis=0) > 1.7)[0],
                    np.where(np.mean(uncon_amplitudes[window[0]:center_CN_spike,:], axis = 0) > 1.7)[0]
                     ]))

# Gaussian blur to unwrapped phases
sigma = 25
spk_uncon_phi_unwrap = np.array( gaussian_filter(spk_uncon_phi_unwrap[:,uncon_include_idx], axes= 0, sigma=sigma))
spk_uncon_phi = spk_uncon_phi[:, uncon_include_idx]
uncon_phi_unwrap = np.array( gaussian_filter(uncon_phi_unwrap[:,uncon_include_idx], axes= 0, sigma=sigma) )
uncon_phi = uncon_phi[:,uncon_include_idx]



con_include_idx = np.unique(np.concatenate([
                   assess_phase_linearity(spk_con_phi_unwrap, time_window = (window[0], center_CN_spike), time=time, r_sqr_thresh = r_sqr_thresh),
                   assess_phase_linearity(con_phi_unwrap, time_window = (window[0], center_CN_spike), time=time, r_sqr_thresh = r_sqr_thresh  ),
                   ]))

# Gaussian blur to unwrapped phases
spk_con_phi_unwrap = gaussian_filter(spk_con_phi_unwrap[:,con_include_idx], axes = 0 , sigma = sigma)
spk_con_phi = spk_con_phi[:, con_include_idx]
con_phi_unwrap = gaussian_filter( con_phi_unwrap[:,con_include_idx], axes = 0, sigma = sigma)
con_phi = con_phi[:,con_include_idx]

n_trials_con =  len(con_include_idx)
n_trials_uncon = len(uncon_include_idx)


uncon_IO_traces_stacked= uncon_IO_traces_stacked[:, uncon_include_idx]
con_IO_traces_stacked = con_IO_traces_stacked[:, con_include_idx]
spk_uncon_IO_traces_stacked = spk_uncon_IO_traces_stacked[:, uncon_include_idx]
spk_con_IO_traces_stacked = spk_con_IO_traces_stacked[:, con_include_idx]


print(f'Number of trials included, unconnected = {n_trials_uncon}')
print(f'Number of trials included, connected = {n_trials_con}')

# Determine phase at time of CN spike # unwrap_phi_at_spk_uncon_linfit = get_linfit_measure(phi_signal= uncon_phi_unwrap, time=time, idx_measure=center_CN_spike)
unwrap_phi_at_spk_uncon = uncon_phi_unwrap[center_CN_spike,:]
phi_at_spk_uncon = (unwrap_phi_at_spk_uncon + np.pi) % (2 * np.pi)  # Needs to be in range 0-2pi

uncon_phase_sort = np.argsort(phi_at_spk_uncon)
phi_at_spk_uncon= phi_at_spk_uncon[uncon_phase_sort]


unwrap_phi_at_spk_con = con_phi_unwrap[center_CN_spike,:]
phi_at_spk_con= (unwrap_phi_at_spk_con + np.pi) % (2 * np.pi)

con_phase_sort = np.argsort(phi_at_spk_con)
phi_at_spk_con = phi_at_spk_con[con_phase_sort]


# Determine time of 1-2 baseline cycles later
n_cycles = 0.5

target_phase_2cycles = unwrap_phi_at_spk_uncon + 2 * np.pi * n_cycles
idx_measure_uncon= np.array([np.where(uncon_phi_unwrap[center_CN_spike+1:, trial] >= target_phase_2cycles[trial])[0][0] for trial in np.arange(n_trials_uncon)]) +center_CN_spike+1
idx_measure_uncon_nxt= np.array([np.where((uncon_phi_unwrap[center_CN_spike+1:, trial] + np.pi)% (np.pi *2) >= (np.pi *1.9))[0][0] for trial in np.arange(n_trials_uncon)]) +center_CN_spike+1

target_phase_2cycles = unwrap_phi_at_spk_con + 2 * np.pi * n_cycles
idx_measure_con = np.array([np.where(con_phi_unwrap[center_CN_spike+1:,trial] >= target_phase_2cycles[trial])[0][0] for trial in np.arange(n_trials_con)] ) +center_CN_spike+1
idx_measure_con_nxt= np.array([np.where((con_phi_unwrap[center_CN_spike+1:, trial] + np.pi)% (np.pi *2) >= (np.pi *1.9))[0][0] for trial in np.arange(n_trials_con)]) +center_CN_spike+1



# Measurement 1: Get phase difference between baseline and spike
uncon_measure = np.array([spk_uncon_phi_unwrap[idx_measure_uncon[trial],trial] - uncon_phi_unwrap[idx_measure_uncon[trial],trial] for trial in np.arange(n_trials_uncon)])
uncon_delta_phase = uncon_measure[uncon_phase_sort] % (np.pi * 2) -np.pi
uncon_delta_phase_without = uncon_measure[uncon_phase_sort]

uncon_measure_nxt = np.array([spk_uncon_phi_unwrap[idx_measure_uncon[trial],trial] - uncon_phi_unwrap[idx_measure_uncon[trial],trial] for trial in np.arange(n_trials_uncon)])

con_measure = np.array([spk_con_phi_unwrap[idx_measure_con[trial],trial]- con_phi_unwrap[idx_measure_con[trial],trial] for trial in np.arange(n_trials_con)])
con_delta_phase = con_measure[con_phase_sort] % (np.pi * 2) -np.pi
con_delta_phase_without = con_measure[con_phase_sort]

uncon_delta_phase_baseline_offset = np.array([spk_uncon_phi_unwrap[center_CN_spike-400,trial] - uncon_phi_unwrap[center_CN_spike - 400,trial] for trial in np.arange(n_trials_uncon)])
con_delta_phase_baseline_offset = np.array([spk_con_phi_unwrap[center_CN_spike - 400,trial]- con_phi_unwrap[center_CN_spike - 400,trial] for trial in np.arange(n_trials_con)])

# Fit fourier series:  Δφ(θ) ≈ a0 + Σ_k (a_k sin(kθ) + b_k cos(kθ))
# to binned and unbinned data

K_max = 2 # order of fourier
theta_grid, PCR_d_phi = fit_fourier_PRC( theta = phi_at_spk_uncon, d_phi = uncon_delta_phase, K_max= K_max)
con_theta_grid, con_PCR_d_phi = fit_fourier_PRC( theta = phi_at_spk_con, d_phi = con_delta_phase, K_max= K_max)






In [ ]:
def mark_value_at_vline(ax, x, time, trace_1d, offset=0, fmt="{:.2f}", **text_kwargs):
    """
    ax        : matplotlib Axes
    x         : x-position of vertical line
    time      : 1D time array
    trace_1d  : 1D signal (same length as time)
    offset    : vertical offset used in plotting
    fmt       : format string for value
    """
    idx = np.argmin(np.abs(time - x))
    y_true = trace_1d[idx]          # true value (no offset)
    y_plot = y_true + offset        # where the curve is actually drawn

    # point at intersection
    ax.scatter(x, y_plot, color='k', s=20, zorder=5)

    # annotate with TRUE value
    ax.annotate(fmt.format(y_true),
                xy=(x, y_plot),
                xytext=(4, 4),
                textcoords="offset points",
                va="bottom",
                fontsize=8,
                **text_kwargs)

    return y_true

# Find outliers and see what is happenning at CN spike disturbance
uncon_outlier_idxs = np.where(uncon_measure  > (5* np.pi))[0]
con_outlier_idxs = np.where(con_measure > (5 * np.pi))[0]

print(len(uncon_outlier_idxs))
print(len(con_outlier_idxs))

fig, axes_uncon = plt.subplots(len(uncon_outlier_idxs), 3, figsize=(13, 10),  constrained_layout=True)
for i, io in enumerate(uncon_outlier_idxs):
    # ---- column 0: wrapped ----
   
    axes_uncon[i, 0].plot(time, spk_uncon_phi[:, io],         label='CN spike')
    axes_uncon[i, 0].plot(time, uncon_phi[:, io] + 7,         label='baseline')
    

    x_grey = DCN_spike_time[1]
    x_red  = time[idx_measure_uncon[io]]

    axes_uncon[i, 0].axvline(x_grey, alpha=0.8, color='grey', linewidth=2)
    axes_uncon[i, 0].axvline(x_red,  alpha=0.8, color='red',  linewidth=1)

    # CN spike (no offset), both vertical lines
    mark_value_at_vline(axes_uncon[i, 0], x_red,  time, spk_uncon_phi[:, io], offset=0, ha="left" )
    mark_value_at_vline(axes_uncon[i, 0], x_grey, time, spk_uncon_phi[:, io], offset=0, ha="right")

    # baseline (+7 offset), both vertical lines
    mark_value_at_vline(axes_uncon[i, 0], x_red,  time, uncon_phi[:, io], offset=7, ha="left")
    mark_value_at_vline(axes_uncon[i, 0], x_grey, time, uncon_phi[:, io], offset=7, ha="right")

    axes_uncon[i, 0].set_xlim(2200, 2800)
    axes_uncon[i, 0].legend(loc='lower left')
    axes_uncon[i,0].set_title(rf'$\Delta \phi$ = {uncon_delta_phase_without[io]} / {uncon_delta_phase[io]}  ')


    # ---- column 1: unwrapped ----
    
    axes_uncon[i, 1].plot(time, spk_uncon_phi_unwrap[:, io],     label='CN spike')
    axes_uncon[i, 1].plot(time, uncon_phi_unwrap[:, io] + 250,   label='baseline')

    axes_uncon[i, 1].axvline(x_grey, alpha=0.8, color='grey', linewidth=2)
    axes_uncon[i, 1].axvline(x_red,  alpha=0.8, color='red',  linewidth=1)

    # CN spike unwrapped (no offset)
    mark_value_at_vline(axes_uncon[i, 1], x_red,  time, spk_uncon_phi_unwrap[:, io], offset=0, ha="left")
    mark_value_at_vline(axes_uncon[i, 1], x_grey, time, spk_uncon_phi_unwrap[:, io], offset=0, ha="right")

    # baseline unwrapped (+250 offset)
    mark_value_at_vline(axes_uncon[i, 1], x_red,  time, uncon_phi_unwrap[:, io], offset=250, ha="left")
    mark_value_at_vline(axes_uncon[i, 1], x_grey, time, uncon_phi_unwrap[:, io], offset=250, ha="right")

    axes_uncon[i, 1].set_xlim(2200, 2800)
    axes_uncon[i, 1].set_ylim(200, 700)
    axes_uncon[i, 1].legend(loc='lower left')
    axes_uncon[i,1].set_title(rf'$\Delta \phi$ = {uncon_delta_phase_without[io]} / {uncon_delta_phase[io]} ')


    # ---- column 2: io mebrane traces ------------
    axes_uncon[i, 2].plot(time, spk_uncon_IO_traces_stacked[:, io], label='CN spike')
    axes_uncon[i, 2].plot(time, uncon_IO_traces_stacked[:, io] + 9, label='baseline' )
    axes_uncon[i, 2].legend()
    axes_uncon[i, 2].set_xlim(2200, 2800)
    axes_uncon[i, 2].axvline(x_grey, alpha=0.8, color='grey', linewidth=2)
    axes_uncon[i, 2].axvline(x_red,  alpha=0.8, color='red',  linewidth=1)

fig, axes_con = plt.subplots(len(con_outlier_idxs), 3, figsize=(13, 10),  constrained_layout=True)
for i, io in enumerate(con_outlier_idxs):
    # ---- column 0: wrapped ----
    axes_con[i, 0].plot(time, spk_con_phi[:, io],        label='CN spike')
    axes_con[i, 0].plot(time, con_phi[:, io] + 7,        label='baseline')

    x_grey = DCN_spike_time[1]
    x_red  = time[idx_measure_con[io]]

    axes_con[i, 0].axvline(x_grey, alpha=0.8, color='grey', linewidth=2)
    axes_con[i, 0].axvline(x_red,  alpha=0.8, color='red',  linewidth=1)

    mark_value_at_vline(axes_con[i, 0], x_red,  time, spk_con_phi[:, io], offset=0, ha="left")
    mark_value_at_vline(axes_con[i, 0], x_grey, time, spk_con_phi[:, io], offset=0, ha="right")
    mark_value_at_vline(axes_con[i, 0], x_red,  time, con_phi[:, io], offset=7, ha="left")
    mark_value_at_vline(axes_con[i, 0], x_grey, time, con_phi[:, io], offset=7,ha="right")

    axes_con[i, 0].set_xlim(2200, 2800)
    axes_con[i, 0].legend(loc='lower left')
    axes_con[i,0].set_title(rf'$\Delta \phi$ = {con_delta_phase_without[io]} / {con_delta_phase[io]} ')

    # ---- column 1: unwrapped ----
    axes_con[i, 1].plot(time, spk_con_phi_unwrap[:, io],     label='CN spike')
    axes_con[i, 1].plot(time, con_phi_unwrap[:, io] + 250,   label='baseline')

    axes_con[i, 1].axvline(x_grey, alpha=0.8, color='grey', linewidth=2)
    axes_con[i, 1].axvline(x_red,  alpha=0.8, color='red',  linewidth=1)

   
    mark_value_at_vline(axes_con[i, 1], x_red,  time, spk_con_phi_unwrap[:, io], offset=0, ha="left")
    mark_value_at_vline(axes_con[i, 1], x_grey, time, spk_con_phi_unwrap[:, io], offset=0, ha="right")
    mark_value_at_vline(axes_con[i, 1], x_red,  time, con_phi_unwrap[:, io], offset=250, ha="left")
    mark_value_at_vline(axes_con[i, 1], x_grey, time, con_phi_unwrap[:, io], offset=250, ha="right")

    axes_con[i, 1].set_xlim(2200, 2800)
    axes_con[i, 1].set_ylim(200, 700)
    axes_con[i, 1].legend()
    axes_con[i,1].set_title(rf'$\Delta \phi$ = {con_delta_phase_without[io]} / {con_delta_phase[io]}')


    # ---- column 2: io mebrane traces ------------
    axes_con[i, 2].plot(time, spk_con_IO_traces_stacked[:, io], label='CN spike')
    axes_con[i, 2].plot(time, con_IO_traces_stacked[:, io] + 9, label='baseline' )
    axes_con[i, 2].legend(loc='lower left')
    axes_con[i, 2].set_xlim(2200, 2800)
    axes_con[i, 2].axvline(x_grey, alpha=0.8, color='grey', linewidth=2)
    axes_con[i, 2].axvline(x_red,  alpha=0.8, color='red',  linewidth=1)




In [ ]:
# Diagnostic plots


# Check linear fitting
unwrap_phi_at_spk_uncon_linfit = get_linfit_measure(phi_signal= uncon_phi_unwrap, time=time, idx_measure=center_CN_spike)

# Check PSD of detrended data (multi-taper)
fig, ax = plot_multitaper_PSD(data_detrend, fs=Fs, labels=['CN inhibition unconnected', 'CN inhibition connected', 'Baseline unconnected', 'Baseline connected'])
ax.set_xlim(0,20)



# Check wrapped and unwrapped phases after filtering out non-periodic signals

# Everything together
fig, (ax1, ax2) = plt.subplots(2)
ax1.set_title('Unconnected all unwrapped phases')
ax1.plot(time, np.mean(spk_uncon_phi_unwrap, axis=1), color='midnightblue')
# ax1.fill_between(
#     time,
#     np.percentile(spk_uncon_phi_unwrap, 25, axis=1),
#     np.percentile(spk_uncon_phi_unwrap, 75, axis=1),
#     color='midnightblue', alpha=0.5, zorder=1
# )
ax1.plot(time, np.mean(uncon_phi_unwrap, axis =1), color= 'orange')
# ax1.fill_between(
#     time,
#     np.percentile(uncon_phi_unwrap, 25, axis=1),
#     np.percentile(uncon_phi_unwrap, 75, axis=1),
#     color='orange', alpha=0.5, zorder=1
# )
ax1.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2)
ax1.set_xlim(2480, 2590)
ax1.set_ylim(300, 370)


ax2.set_title('Connected all unwrapped phases')
ax2.plot(time, np.mean(spk_con_phi_unwrap, axis=1), color='midnightblue')
# ax2.fill_between(
#     time,
#     np.percentile(spk_con_phi_unwrap, 25, axis=1),
#     np.percentile(spk_con_phi_unwrap, 75, axis=1),
#     color='midnightblue', alpha=0.5, zorder=1
# )
ax2.plot(time, np.mean(con_phi_unwrap, axis=1), color= 'orange')
# ax2.fill_between(
#     time,
#     np.percentile(con_phi_unwrap, 25, axis=1),
#     np.percentile(con_phi_unwrap, 75, axis=1),
#     color='orange', alpha=0.5, zorder=1
# )
ax2.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2)
ax2.set_xlim(2480, 2590)
ax2.set_ylim(300, 370)



# Wrapped phases around CN_spike
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, sharex=True, figsize= (7,12))

offset_unwrap= 90
offset_wrap = 10
two_ios_to_plot = [78, 234]



ax1.set_title("Unconnexted wrapped phases ")
ax1.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2)
for i,io_idx in enumerate(two_ios_to_plot):
    ax1.plot(time, spk_uncon_phi[:, io_idx] + offset_wrap * i *2, label= f'{io_idx} (CN spike)')
    ax1.plot(time, uncon_phi[:, io_idx] + offset_wrap * i *2 + offset_wrap, label= f'{io_idx} (baseline)')
ax1.set_xlim(windowed_time[0], windowed_time[-1])
ax1.legend(loc='upper left', bbox_to_anchor=(1.05, 1), borderaxespad=0)

ax2.set_title("Unconnexted unwrapped phases")
ax2.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2)
for i,io_idx in enumerate(two_ios_to_plot):
    ax2.plot(time, spk_uncon_phi_unwrap[:, io_idx] + offset_unwrap * i*2, label= f'{io_idx} (CN spike)')
    ax2.plot(time, uncon_phi_unwrap[:, io_idx] + offset_unwrap * i *2+ offset_unwrap, label= f'{io_idx} (baseline)')
ax2.set_xlim(windowed_time[0], windowed_time[-1])
ax2.set_ylim(200,700)
ax2.legend(loc='upper left', bbox_to_anchor=(1.05, 1), borderaxespad=0)

ax3.set_title("Connexted wrapped phases")
ax3.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2)
for i,io_idx in enumerate(two_ios_to_plot):
    ax3.plot(time, spk_con_phi[:, io_idx] + offset_wrap * i *2, label= f'{io_idx} (CN spike)')
    ax3.plot(time, con_phi[:, io_idx] + offset_wrap * i *2+ offset_wrap, label= f'{io_idx} (baseline)')
ax3.set_xlim(windowed_time[0], windowed_time[-1])
ax3.legend(loc='upper left', bbox_to_anchor=(1.05, 1), borderaxespad=0)

ax4.set_title("Connexted unwrapped phases ")
ax4.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2)
for i,io_idx in enumerate(two_ios_to_plot):
    ax4.plot(time, spk_con_phi_unwrap[:, io_idx] + offset_unwrap * i *2, label= f'{io_idx} (CN spike)')
    ax4.plot(time, con_phi_unwrap[:, io_idx] + offset_unwrap * i *2 + offset_unwrap, label= f'{io_idx} (baseline)')
ax4.set_xlim(windowed_time[0], windowed_time[-1])
ax4.set_ylim(200,700)
ax4.legend(loc='upper left', bbox_to_anchor=(1.05, 1), borderaxespad=0)

# Check PRCs with and without mod 2pi
fig, (ax1, ax2, ax3, ax4, ax5, ax6) = plt.subplots(6, sharex=True, figsize = (7,16))
ax1.set_title(f'Unconnected IO network PRC')
ax1.scatter(phi_at_spk_uncon, uncon_delta_phase,  s=8, alpha=0.3 )
ax1.plot(theta_grid, PCR_d_phi)
ax1.set_xlabel(r'$\phi$ (rad)')
ax1.set_ylabel(r'$\Delta \phi$')
ax1.set_ylim(-8, 8)
ax1.axhline(0, color='black', linewidth=1, linestyle='--')


ax2.set_title(f'Connected IO network PRC')
ax2.scatter(phi_at_spk_con, con_delta_phase,  s=8, alpha=0.3 )
ax2.plot(con_theta_grid, con_PCR_d_phi)
ax2.set_xlabel(r'$\phi$ (rad)')
ax2.set_ylabel(r'$\Delta \phi$')
ax2.set_ylim(-8, 8)
ax2.axhline(0, color='black', linewidth=1, linestyle='--')


ax3.set_title(f'Unconnected IO network PRC')
ax3.scatter(phi_at_spk_uncon, uncon_delta_phase_without,  s=8, alpha=0.3 )
# ax1.plot(theta_grid, PCR_d_phi)
ax3.set_xlabel(r'$\phi$ (rad)')
ax3.set_ylabel(r'$\Delta \phi$')
ax3.set_ylim(-8, 8)
ax3.axhline(0, color='black', linewidth=1, linestyle='--')


ax4.set_title(f'Connected IO network PRC')
ax4.scatter(phi_at_spk_con, con_delta_phase_without,  s=8, alpha=0.3 )
# ax2.plot(con_theta_grid, con_PCR_d_phi)
ax4.set_xlabel(r'$\phi$ (rad)')
ax4.set_ylabel(r'$\Delta \phi$')
ax4.set_ylim(-8, 8)
ax4.axhline(0, color='black', linewidth=1, linestyle='--')

ax5.set_title(f'Unconnected IO network PRC, baseline control')
ax5.scatter(np.random.rand(uncon_delta_phase_baseline_offset.shape[0]), uncon_delta_phase_baseline_offset,  s=8, alpha=0.3 )
# ax3.plot(con_theta_grid, con_PCR_d_phi_mod)
ax5.set_xlabel(r'$\phi$ (rad)')
ax5.set_ylabel(r'$\Delta \phi$')
ax5.set_ylim(-np.pi-0.3, np.pi+0.3)
ax5.axhline(0, color='black', linewidth=1, linestyle='--')

ax6.set_title(f'Connected IO network PRC, baseline control')
ax6.scatter(np.random.rand(con_delta_phase_baseline_offset.shape[0]), con_delta_phase_baseline_offset,  s=8, alpha=0.3 )
# ax4.plot(con_theta_grid, con_PCR_d_phi)
ax6.set_xlabel(r'$\phi$ (rad)')
ax6.set_ylabel(r'$\Delta \phi$')
ax6.set_ylim(-8, 8)
ax6.axhline(0, color='black', linewidth=1, linestyle='--')

In [ ]:
unwrap_phi_at_spk_uncon_linfit = get_linfit_measure(phi_signal= uncon_phi_unwrap, time=time, idx_measure=center_CN_spike)

In [ ]:
# Final figure
fig, ((ax1, ax2),
      (ax3, ax4),
      (ax5, ax6)) = plt.subplots(
    3, 2,
    figsize=(10, 8),
    gridspec_kw={'height_ratios': [2, 1, 2]}, # one height ratio per row
    sharey= False
)


random_run = np.random.randint(n_runs)
n_ios = 6
offset_step = 80.0




# --------------------
# Unconnected
# --------------------
IO_traces_subs = np.column_stack([
    spk_single_IO_traces_unconnected[random_run, :, io]
    for io in np.arange(n_ios)
])  # shape: (time, n_io_plot)

IO_phases_subs, amp = filter_and_Hilbert(IO_traces_subs, a = uncon_freqrange[0], b = uncon_freqrange[1], Fs=Fs)

for io_idx in np.arange(n_ios):
    base = offset_step * io_idx
    IO_trace = IO_traces_subs[:, io_idx]
    ax1.plot(windowed_time, IO_trace +  base)
    phase_trace =  IO_phases_subs[:, io_idx]

ax1.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2)
ax1.set_xlabel('time (s)')
ax1.set_ylabel('potential (mV)')
ax1.set_yticks([])
ax1.set_xlim(windowed_time[0], windowed_time[-1])
ax1.set_axis_off()
ax1.set_title('Unconnected IO membrane potential traces')

# --------------------
# Connected
# --------------------


IO_traces_subs = np.column_stack([
    spk_single_IO_traces_connected[random_run, :, io]
    for io in np.arange(n_ios)
])  # shape: (time, n_io_plot)

IO_phases_subs, amp = filter_and_Hilbert(IO_traces_subs, a = con_freqrange[0], b = con_freqrange[1], Fs=Fs)

for io_idx in np.arange(n_ios):
    IO_trace = IO_traces_subs[:, io_idx]
    phase_trace =  IO_phases_subs[:, io_idx]
    ax2.plot(windowed_time, IO_trace + offset_step * io_idx)
    # ax2.plot(windowed_time, IO_trace + offset_step2 * io_idx, color= 'grey')

ax2.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2)
ax2.set_xlabel('time (s)')
ax2.set_ylabel('potential (mV)')
ax2.set_yticks([])
ax2.set_xlim(windowed_time[0], windowed_time[-1])
ax2.set_axis_off()
ax2.set_title('Connected IO membrane potential traces')

# --------------------
# Median unconnected
# --------------------
ax3.plot(windowed_time, np.median(spk_uncon_IO_traces_stacked, axis=1),
         color='midnightblue', linewidth=2, zorder=2)
ax3.fill_between(
    windowed_time,
    np.percentile(spk_uncon_IO_traces_stacked, 25, axis=1),
    np.percentile(spk_uncon_IO_traces_stacked, 75, axis=1),
    color='midnightblue', alpha=0.5, zorder=1
)
ax3.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2, zorder=0)
ax3.set_xlabel('time (ms)')
ax3.set_ylabel('Membrane potential (mV)')
# ax3.set_axis_off()

# --------------------
# Median connected
# --------------------
ax4.plot(windowed_time, np.median(spk_con_IO_traces_stacked, axis=1),
         color='midnightblue', linewidth=2, zorder=2)
ax4.fill_between(
    windowed_time,
    np.percentile(spk_con_IO_traces_stacked, 25, axis=1),
    np.percentile(spk_con_IO_traces_stacked, 75, axis=1),
    color='midnightblue', alpha=0.5, zorder=1
)
ax4.axvline(DCN_spike_time[1], alpha=0.8, color='grey', linewidth=2, zorder=0)
ax4.set_xlabel('time (ms)')
ax4.set_ylabel('Membrane potential (mV)')
# ax4.set_axis_off()


# -----------------
# PRC unconnected
# ------------------

xticks = np.linspace(0, 2*np.pi, 5)
yticks = np.arange(-4*np.pi, 4.1*np.pi, np.pi)



ax5.scatter(phi_at_spk_uncon, uncon_delta_phase,  s=8, alpha=0.3 )
ax5.plot(theta_grid, PCR_d_phi,  linewidth=3)
ax5.set_xlabel(r'$\phi$ ')
ax5.set_xticks(xticks)
ax5.set_xticklabels(['0', r'$\frac{\pi}{2}$', r'$\pi$',
                        r'$\frac{3\pi}{2}$', r'$2\pi$'])

ax5.set_ylabel(r'$\Delta \phi$ (rad)')
ax5.set_ylim(-4*np.pi, 4*np.pi)
ax5.set_yticks(yticks)
ax5.set_yticklabels([
    r'$-4\pi$', r'$-3\pi$', r'$-2\pi$', r'$-\pi$',
    r'$0$', r'$\pi$', r'$2\pi$', r'$3\pi$', r'$4\pi$'
])
ax5.axhline(0, color='black', linewidth=1, linestyle='--')



# ----------------
# PRC connected
# ------------------

ax6.scatter(phi_at_spk_con, con_delta_phase,  s=8, alpha=0.3 )
ax6.plot(con_theta_grid, con_PCR_d_phi, linewidth=3)
ax6.set_xlabel(r'$\phi$ ')
ax6.set_xticks(xticks)
ax6.set_xticklabels(['0', r'$\frac{\pi}{2}$', r'$\pi$',
                        r'$\frac{3\pi}{2}$', r'$2\pi$'])

ax6.set_ylabel(r'$\Delta \phi$ (rad)')
ax6.set_ylim(-4*np.pi, 4*np.pi)
ax6.set_yticks(yticks)
ax6.set_yticklabels([
    r'$-4\pi$', r'$-3\pi$', r'$-2\pi$', r'$-\pi$',
    r'$0$', r'$\pi$', r'$2\pi$', r'$3\pi$', r'$4\pi$'
])
ax6.axhline(0, color='black', linewidth=1, linestyle='--')



plt.tight_layout()

In [ ]:
# Measurement 2: Difference in time until next spike after CN input

# Sort spiking data in the same way phase data is sorted
# uncon_spk_io_spiking = sort_spiking_data(IO_spike_dat = spike_dat['io.spike'], run_idx =spk_unconnected_run_idx, trial_idx= uncon_include_idx, argsort_idx= uncon_phase_sort)
# uncon_io_spiking = sort_spiking_data(IO_spike_dat = dat['io.spike'], run_idx = unconnected_run_idx, trial_idx = uncon_include_idx, argsort_idx = uncon_phase_sort)
#
# con_spk_io_spiking = sort_spiking_data(IO_spike_dat = spike_dat['io.spike'], run_idx =spk_connected_run_idx, trial_idx= con_include_idx, argsort_idx= con_phase_sort)
# con_io_spiking = sort_spiking_data(IO_spike_dat = dat['io.spike'], run_idx = connected_run_idx, trial_idx = con_include_idx, argsort_idx = con_phase_sort)


# # Find first IO spike after CN spike
# uncon_spk_first_spike_idx, uncon_first_spike_idx, valid_trials_uncon = find_first_spike(IO_spike_dat= uncon_spk_io_spiking, IO_baseline_dat = uncon_io_spiking, center = center_CN_spike)
# t_spike_after_uncon_spk = time[center_CN_spike+1+uncon_spk_first_spike_idx]
# t_spike_after_uncon = time[center_CN_spike+1+uncon_first_spike_idx]
#
#
# con_spk_first_spike_idx, con_first_spike_idx, valid_trials_con = find_first_spike(IO_spike_dat= con_spk_io_spiking, IO_baseline_dat = con_io_spiking, center = center_CN_spike)
# t_spike_after_con_spk = time[center_CN_spike+1+con_spk_first_spike_idx]
# t_spike_after_con = time[center_CN_spike+1+con_first_spike_idx]
#
#
# # Calculate spike time difference
# valid_phi_at_spk_uncon= phi_at_spk_uncon[valid_trials_uncon]
# uncon_delta_t_spike = t_spike_after_uncon_spk-t_spike_after_uncon
#
# valid_phi_at_spk_con= phi_at_spk_con[valid_trials_con]
# con_delta_t_spike = t_spike_after_con_spk-t_spike_after_con

In [ ]:
# Calculate average over phase bins
# Sort measurements in bins and get average/median per bin
bin_edges = np.histogram_bin_edges(phi_at_spk_uncon, bins = 20, range = (0, 2 * np.pi))
idx_per_bin = np.digitize(phi_at_spk_uncon, bin_edges)
bins = np.unique(idx_per_bin)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

binned_measurements = {f'{bin}': uncon_delta_phase[idx_per_bin == bin]for bin in bins}
binned_mean = np.array([
    np.mean(binned_measurements[f'{bin}']) for bin in bins
])
binned_std = np.array([
    np.std(binned_measurements[f'{bin}']) for bin in bins
])

con_bin_edges = np.histogram_bin_edges(phi_at_spk_con, bins = 20, range = (0, 2 * np.pi))
con_idx_per_bin = np.digitize(phi_at_spk_con, bin_edges)
con_bins = np.unique(idx_per_bin)
con_bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

con_binned_measurements = {f'{cbin}': con_delta_phase[con_idx_per_bin == cbin]for cbin in con_bins}
con_binned_mean = np.array([
    np.mean(con_binned_measurements[f'{cbin}']) for cbin in con_bins
])
con_binned_std = np.array([
    np.std(con_binned_measurements[f'{cbin}']) for cbin in con_bins
])

bin_theta_grid, bin_PCR_d_phi = fit_fourier_PRC( theta = bin_centers, d_phi = binned_mean, K_max= K_max)

con_bin_theta_grid, con_bin_PCR_d_phi = fit_fourier_PRC( theta = con_bin_centers, d_phi = con_binned_mean, K_max= K_max)